In [ ]:
from pathlib import Path
import random
import yaml
import torch
import kagglehub
from ultralytics import YOLO

In [ ]:

from pathlib import Path
import random
import kagglehub
import yaml
from ultralytics import YOLO

# 1. Télécharger le dataset
dataset_dir = (
    Path(kagglehub.dataset_download(
        "ashfakyeafi/road-vehicle-images-dataset"
    ))
    / "trafic_data"
)

# 2. Lire les informations de data.yaml existant
# Si tu ne veux créer aucun YAML, on définit les classes en Python.
nc = 21
names = [
    "ambulance", "army vehicle", "auto rickshaw", "bicycle",
    "bus", "car", "garbagevan", "human hauler", "minibus",
    "minivan", "motorbike", "pickup", "policecar", "rickshaw",
    "scooter", "suv", "taxi", "three wheelers -CNG-",
    "truck", "van", "wheelbarrow"
]

# 3. Récupérer les images et leurs labels
images_dir = dataset_dir / "train" / "images"
labels_dir = dataset_dir / "train" / "labels"

images = [
    p for p in images_dir.iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    and (labels_dir / f"{p.stem}.txt").exists()
]

# 4. Diviser les chemins : 80% train, 20% test
random.seed(42)
random.shuffle(images)

split_idx = int(len(images) * 0.8)
train_images = images[:split_idx]
test_images = images[split_idx:]

# 5. Utiliser le dossier valid existant
val_images = [
    p for p in (dataset_dir / "valid" / "images").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

print("Train:", len(train_images))
print("Validation:", len(val_images))
print("Test:", len(test_images))


In [ ]:
model = YOLO("yolov8n.pt")
print("Model:", model.info())

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

results = model.train(
    data=str(yaml_path.resolve()),
    epochs=10,
    imgsz=640,
    batch=8,
    device=device,
    project="runs/fine_tune",
    name="yolov8_vehicles"
)

In [ ]:
val_metrics = model.val(split="val")
print("Val mAP50-95:", val_metrics.box.map)
print("Val mAP50:", val_metrics.box.map50)

In [ ]:
test_metrics = model.val(split="test")
print("Test mAP50-95:", test_metrics.box.map)
print("Test mAP50:", test_metrics.box.map50)

In [ ]:
if test_images:
    preds = model.predict(source=test_images[0], conf=0.25, save=True)
    print("Detections on test sample:", len(preds[0].boxes))